# `whosampled-brainz.jsonl` -> versao enxuta para o grafo

Reduz cada lado (`from` / `to`) de cada relacao a **um candidato representativo + um conjunto
unico de tags**, filtrando os candidatos do MusicBrainz pelo ano da faixa no WhoSampled.

**Entrada**
- `notebooks/results/whosampled-brainz.jsonl.jsonl` (3,27 GB, 155.646 relacoes) -- saida da
  mineracao (`scripts/mine_musicbrainz.py`), com ate 50 candidatos por lado, cada um carregando
  releases, release groups, similaridades e quatro listas de tags.
- `results/whosampled-data-samples1.json` -- o subset do WhoSampled que originou essa mineracao,
  de onde vem o **ano de referencia** de cada musica.

**Saida** `notebooks/results/whosampled-brainz-clean.jsonl`, uma linha por relacao.

---

## Por que o join e por `key`, e nao por posicao

O arquivo grande foi montado por `notebooks/juntar_batches.ipynb`, que percorre os lotes com
`list(BATCHES_FOLDER.glob('*.jsonl.gz'))` -- e `glob` **nao ordena**. Os lotes entraram fora de
ordem: a linha 0 do jsonl e a key `90710`, enquanto o indice 0 do JSON e a key `580172`, e
**nenhuma** das primeiras 1000 posicoes coincide. A ordem interna de cada lote sobreviveu, a
global nao.

O conjunto de keys, por outro lado, e identico e 1:1 -- 155.646 keys unicas dos dois lados, todas
presentes em `samples1` (intersecao zero com `samples2`). Entao o pareamento e feito **pela key**,
e o notebook estoura `KeyError` se alguma faltar: join silenciosamente errado seria pior que
falhar.

## Qual ano vai com qual lado

O pareamento entre os dois arquivos e fixo e assimetrico:

| lado no jsonl | campo no `whosampled-data-samples1.json` |
|---|---|
| `from` -- a musica que sampleia | `track.year` (mesma entrada de `track.name` e `by`) |
| `to` -- a musica sampleada | `other_track.year` (com `other_track.name` e `other_track_by`) |

E o que `mine_musicbrainz.py` gravou (`entry["track"]` -> `from`, `entry["other_track"]` -> `to`),
conferido nos dados: em 40.000 registros, `music_searched` e `artist_searched` dos dois lados
batem com os campos correspondentes do JSON em 100% dos casos. Cruzar os lados compararia o ano
da musica que sampleia com os candidatos da musica sampleada -- em geral decadas de distancia.

## As duas regras

Aplicadas **de forma totalmente independente** a `from` e a `to`: cada lado tem seu proprio ano de
referencia, seus proprios candidatos, e nenhum candidato ou tag cruza de um lado para o outro.

**A. Filtro por ano.** Mantem os candidatos com `release_year` dentro de +-1 ano da referencia
(`year_filter_status = "exact"`). Se nenhum cair no intervalo, nada e descartado: ficam o(s) de
menor diferenca absoluta, todos os empatados inclusive (`"fallback"`). Candidato **sem**
`release_year` nao concorre -- nem no +-1, nem no fallback; quando *todos* os candidatos do lado
sao sem ano (941 lados em `from`, 508 em `to`), mantem-se todos e marca-se `"no_year"`.

**B. Merge de tags.** Sobre os sobreviventes da regra A, na ordem
`music_tags` > `release_tags` > `release_group_tags` > `artist_tags`: vence o **primeiro nivel com
ao menos um candidato de lista nao-vazia**, e nesse nivel as listas de todos os candidatos
preenchidos sao unidas com dedup. Nenhum nivel preenchido -> lado sem tags.

In [ ]:
from pathlib import Path

REPO   = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SRC    = REPO / "results" / "whosampled-brainz.jsonl"
SUBSET = REPO / "results"   / "whosampled-data-samples.json"
OUT    = REPO / "results" / "whosampled-brainz-clean.jsonl"

# None = arquivo inteiro (155.646 relacoes, alguns minutos).
# Ponha 20_000 para iterar rapido enquanto mexe nas regras.
MAX_LINES = None

# Tolerancia do filtro de ano, em anos, para cada lado (regra A)
YEAR_TOL = 1

# So para a barra de progresso quando MAX_LINES e None
N_RECORDS_TOTAL = 155_646*2

for p in (SRC, SUBSET):
    status = f"{p.stat().st_size / 1e9:5.2f} GB" if p.exists() else "NAO ENCONTRADO"
    print(f"{p.name:<38} {status}")

whosampled-brainz.jsonl                 6.98 GB
whosampled-data-samples.json            0.22 GB


## Anos de referencia

Carrega o subset inteiro (pico de ~1,0 GB de RSS), guarda so `{key: (ano do from, ano do to)}` e
libera o resto -- o dict que sobra sao 155.646 tuplas de dois ints.

`track.year` vem como **string** em 13.668 entradas (`"1995"`); `other_track.year` sempre vem como
int. `as_year` normaliza os dois casos. Nenhum ano e nulo neste subset, mas a funcao devolve
`None` se for -- e a regra A trata esse caso.

In [2]:
import gc
import json


def as_year(value):
    """int(value) tolerante: 13.668 anos de `track` vem como string ('1995')."""
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def load_reference_years(path=None):
    """{key: (ano do `track`, ano do `other_track`)} -- nessa ordem: `from`, depois `to`."""
    path = SUBSET if path is None else path
    with open(path, encoding="utf-8") as f:
        subset = json.load(f)

    years = {}
    for key, actions in subset.items():
        # cada entrada do subset tem exatamente uma acao ('sampled')
        entry = next(iter(actions.values()))
        years[key] = (as_year(entry["track"]["year"]),          # -> lado `from`
                      as_year(entry["other_track"]["year"]))    # -> lado `to`

    del subset
    gc.collect()
    return years


REFERENCE_YEARS = load_reference_years()

n_null = sum(y is None for pair in REFERENCE_YEARS.values() for y in pair)
print(f"{len(REFERENCE_YEARS):,} entradas | anos nulos: {n_null}")
print("exemplo:", *next(iter(REFERENCE_YEARS.items())))

311,292 entradas | anos nulos: 0
exemplo: 580172 (2013, 1981)


## As regras, isoladas

Funcoes puras: recebem a lista de candidatos de **um** lado e devolvem o resultado daquele lado.
Nada aqui olha para o lado oposto.

In [3]:
import collections

TAG_LEVELS = ("music_tags", "release_tags", "release_group_tags", "artist_tags")
LEVEL_NAME = {"music_tags": "music", "release_tags": "release",
              "release_group_tags": "release_group", "artist_tags": "artist"}

# campos do candidato representativo que sobrevivem no arquivo limpo
KEPT_FIELDS = ("database_artist", "database_recording", "artist_gid", "recording_gid",
               "release_year", "combined_similarity",
               "artist_type", "artist_gender",
               "artist_area", "artist_area_type", "artist_country",
               "release_group_types")

# lado sem match: mesma forma, tudo vazio -- assim toda linha do jsonl tem as mesmas chaves
EMPTY_SIDE = {
    **{f: None for f in KEPT_FIELDS},
    "release_group_types": [],
    "n_candidates_original": 0,
    "n_candidates": 0,
    "year_filter_status": None,
    "years_considered": [],
    "tags": [],
    "tags_source_level": None,
}


def year_filter(candidates, reference_year, tol=YEAR_TOL):
    """Regra A -> (sobreviventes, status).

    Candidato sem `release_year` nao concorre: nem no +-tol, nem no fallback. Medido no arquivo
    inteiro, isso custa quase nada em tags (0,04% dos lados `from` e 0,10% dos `to` ficariam sem
    tags que teriam) e evita inflar `to` de 2,43 para 3,61 candidatos por lado com gravacoes de
    epoca desconhecida.
    """
    if reference_year is None:
        # nao acontece em samples1; existe para o notebook rodar em outros subsets
        return candidates, "no_ref_year"

    pool = [c for c in candidates if c.get("release_year") is not None]
    if not pool:
        # nenhum candidato datado: nao ha o que filtrar, o lado inteiro fica
        return candidates, "no_year"

    near = [c for c in pool if abs(c["release_year"] - reference_year) <= tol]
    if near:
        return near, "exact"

    # ninguem dentro da janela: fica o mais proximo, e TODOS os empatados junto
    closest = min(abs(c["release_year"] - reference_year) for c in pool)
    return [c for c in pool if abs(c["release_year"] - reference_year) == closest], "fallback"


def merge_tags(candidates):
    """Regra B -> (nome do nivel, tags). Primeiro nivel com alguem preenchido vence.

    Um unico candidato preenchido no nivel cai no mesmo caminho da uniao -- o `dict.fromkeys`
    deduplica preservando a ordem em que as tags aparecem.
    """
    for level in TAG_LEVELS:
        filled = [c[level] for c in candidates if c.get(level)]
        if not filled:
            continue
        return LEVEL_NAME[level], list(dict.fromkeys(t for tags in filled for t in tags))
    return None, []


def clean_side(side, reference_year, stats):
    """Aplica A e B a UM lado e devolve o dict enxuto, acumulando as estatisticas."""
    stats["sides"] += 1

    base = {
        "match_found": bool(side["database_results"]),
        "artist_searched": side["artist_searched"],
        "music_searched": side["music_searched"],
        "reference_year": reference_year,
    }

    candidates = side["database_results"]
    if not candidates:
        # sem match: nada a filtrar. Nao interrompe o lado oposto nem a entrada.
        stats["no_match"] += 1
        return {**base, **EMPTY_SIDE}

    stats["matched"] += 1
    stats["candidates"] += len(candidates)

    survivors, status = year_filter(candidates, reference_year)
    stats["survivors"] += len(survivors)
    stats["status_" + status] += 1

    level, tags = merge_tags(survivors)
    stats["level_" + (level or "none")] += 1
    if not tags:
        stats["no_tags"] += 1

    # `database_results` ja vem ordenado por combined_similarity desc; `max` devolve o primeiro
    # maximal, entao empates continuam respeitando essa ordem
    best = max(survivors, key=lambda c: c.get("combined_similarity") or 0.0)

    return {
        **base,
        **{f: best.get(f) for f in KEPT_FIELDS},
        "n_candidates_original": len(candidates),
        "n_candidates": len(survivors),
        "year_filter_status": status,
        # auditoria da regra A: o ano de TODOS os candidatos originais, nulos inclusive
        "years_considered": [c.get("release_year") for c in candidates],
        "tags": tags,
        "tags_source_level": level,
    }

## Limpeza

Streaming linha a linha: o pico de memoria e um registro mais o dict de anos, entao os 3,27 GB
nunca entram na RAM.

In [4]:
from tqdm.auto import tqdm


def clean_file(src=None, out=None, max_lines=None):
    """Le o jsonl bruto e grava o enxuto. Devolve (n registros, estatisticas por lado)."""
    src = SRC if src is None else src
    out = OUT if out is None else out
    max_lines = MAX_LINES if max_lines is None else max_lines

    stats = {"from": collections.Counter(), "to": collections.Counter()}
    n_records = 0

    with open(src, encoding="utf-8") as f_in, open(out, "w", encoding="utf-8") as f_out:
        progress = tqdm(f_in, total=max_lines or N_RECORDS_TOTAL,
                        desc="relacoes", unit=" rel", smoothing=0.01)
        for line in progress:
            if max_lines is not None and n_records >= max_lines:
                break

            record = json.loads(line)
            key = record["key"]
            # KeyError aqui = join quebrado; melhor estourar que gravar ano errado
            year_from, year_to = REFERENCE_YEARS[key]

            for relation_type, relation in record["relations"].items():
                clean = {
                    "key": key,
                    "relation_type": relation_type,
                    # cada lado com O SEU ano: `track` para from, `other_track` para to
                    "from": clean_side(relation["from"], year_from, stats["from"]),
                    "to":   clean_side(relation["to"],   year_to,   stats["to"]),
                }
                f_out.write(json.dumps(clean, ensure_ascii=False) + "\n")

            n_records += 1
        progress.close()

    return n_records, stats


N_RECORDS, STATS = clean_file()

print(f"\n{N_RECORDS:,} registros -> {OUT.name}")
print(f"{SRC.stat().st_size / 1e9:.2f} GB -> {OUT.stat().st_size / 1e9:.2f} GB "
      f"({100 * OUT.stat().st_size / SRC.stat().st_size:.1f}% do original)")

/home/pedro/Documents/mestrado/sample-detection/sample_mining/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
relacoes: 100%|██████████| 311292/311292 [02:39<00:00, 1951.07 rel/s]



311,292 registros -> whosampled-brainz-clean.jsonl
6.98 GB -> 0.48 GB (6.9% do original)
